In [1]:
!pip install transformers datasets librosa torch torchaudio opensmile praat-parselmouth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 32.4 MB/s eta 0:00:00


In [2]:
# Cell 1 — imports + φόρτωση του ΑΡΧΙΚΟΥ (μη fine-tuned) data2vec ως encoder
import torch
import numpy as np
import os
from transformers import AutoFeatureExtractor, AutoModel
from torch.utils.data import DataLoader

from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

BASE_CHECKPOINT = "facebook/data2vec-audio-base-960h"

base_feature_extractor = AutoFeatureExtractor.from_pretrained(BASE_CHECKPOINT)
base_model = AutoModel.from_pretrained(BASE_CHECKPOINT)   # AutoModel, ΟΧΙ AutoModelForAudioClassification — δεν θέλουμε classification head
base_model.eval()
if torch.cuda.is_available():
    base_model = base_model.to("cuda")

preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  373MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/230 [00:00<?, ?it/s]

[transformers] Data2VecAudioModel LOAD REPORT from: facebook/data2vec-audio-base-960h
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  373MB            

model.safetensors: downloading bytes:           |  0.00B            

In [3]:
# Συναρτήσεις φόρτωσης για όλα τα dataset (ίδιες με τα αντίστοιχα finetuning)
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))
    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename: return "OAF"
        elif "YAF" in filename: return "YAF"
        return "unknown"
    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    return DatasetDict({"train": train_dataset, "test": test_dataset})


def load_ravdess_speaker_independent():
    dataset = load_dataset("narad/ravdess", revision="refs/convert/parquet")["train"]
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

    splitter = GroupShuffleSplit(test_size=0.35, n_splits=1, random_state=42)  # ⚠️ επιβεβαίωσε 0.35
    train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))
    dataset = DatasetDict({
        "train": dataset.select(train_idx),
        "test": dataset.select(test_idx)
    })

    # label_names χρειάζονται για το label_mapping.py (int→string)
    dataset_raw = load_dataset("narad/ravdess", revision="refs/convert/parquet")["train"]
    label_names = dataset_raw.features["labels"].int2str(list(range(dataset_raw.features["labels"].num_classes)))
    dataset.ravdess_label_names = label_names  # βολικό attribute για μετά, όχι απαραίτητο
    return dataset


def load_cremad_speaker_independent():
    dataset = load_dataset("AbstractTTS/CREMA-D", split="train")
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
    def add_speaker(batch):
        batch["actor_id"] = batch["file"].split("_")[0]
        return batch
    dataset = dataset.map(add_speaker)
    splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["actor_id"]))
    return DatasetDict({
        "train": dataset.select(train_idx),
        "test": dataset.select(test_idx)
    })


def load_iemocap_speaker_independent():
    dataset = load_dataset("AbstractTTS/IEMOCAP", split="train")
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
    def merge_and_filter(batch):
        emo = batch["major_emotion"]
        if emo == "excited": emo = "happy"
        batch["emotion_4class"] = emo
        return batch
    dataset = dataset.map(merge_and_filter)
    dataset = dataset.filter(lambda b: b["emotion_4class"] in ["angry", "happy", "neutral", "sad"])
    def add_speaker(batch):
        parts = batch["file"].split("_")
        session = parts[0][:5]
        speaker_gender = parts[-1][0]
        batch["speaker_id"] = f"{session}_{speaker_gender}"
        return batch
    dataset = dataset.map(add_speaker)
    splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))
    return DatasetDict({
        "train": dataset.select(train_idx),
        "test": dataset.select(test_idx)
    })

In [4]:
LOADERS = {
    "TESS":    load_tess_speaker_independent,
    "RAVDESS": load_ravdess_speaker_independent,
    "CREMA-D": load_cremad_speaker_independent,
    "IEMOCAP": load_iemocap_speaker_independent,
}
datasets_raw = {corpus: loader() for corpus, loader in LOADERS.items()}

README.md:   0%|          | 0.00/626 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  184MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2800 [00:00<?, ? examples/s]

... (17 outputs left out)

Generating train split:   0%|          | 0/10039 [00:00<?, ? examples/s]

Map:   0%|          | 0/10039 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10039 [00:00<?, ? examples/s]

Map:   0%|          | 0/6877 [00:00<?, ? examples/s]

In [5]:
# Οι 4 κλάσεις για το cross corpus analysis
COMMON_LABELS = ["angry", "happy", "neutral", "sad"]
COMMON_LABEL2ID = {name: i for i, name in enumerate(COMMON_LABELS)}

#Mapping των ετικετών κάθε συνόλου, σε κοινή ετικές ή none αν είναι εκτός
LABEL_MAPS = {
    "TESS": {
        "angry": "angry", "happy": "happy", "neutral": "neutral", "sad": "sad",
        "disgust": None, "fear": None, "pleasant surprise": None,
    },
    "CREMA-D": {
        "anger": "angry",   #διαφορετικό όνομα από τα άλλα σύνολα
        "happy": "happy", "neutral": "neutral", "sad": "sad",
        "disgust": None, "fear": None,
    },
    "RAVDESS": {
        "angry": "angry", "happy": "happy", "neutral": "neutral", "sad": "sad",
        "calm": None, "fearful": None, "disgust": None, "surprised": None,
    },
    "IEMOCAP": {
    "angry": "angry",
    "happy": "happy",
    "neutral": "neutral",
    "sad": "sad",
    }
}

In [6]:
#Επιστρέφει τους δείκτες εξόδου του μοντέλου που αντιστοιχούν σε μία από τις 4 κοινές κλάσεις.
def get_valid_source_indices(corpus, id2label):
    label_map = LABEL_MAPS[corpus]
    valid = {}
    for idx, raw_label in id2label.items():
        common = label_map.get(raw_label)
        if common is not None:
            valid[idx] = common
    return valid


def remap_prediction(logits_row, corpus, id2label):
    #logits_row: 1D array/tensor, τα raw logits ενός δείγματος από το source model.
    valid_indices = get_valid_source_indices(corpus, id2label)
    restricted = {idx: logits_row[idx] for idx in valid_indices}
    best_idx = max(restricted, key=restricted.get)
    return valid_indices[best_idx]


RAW_LABEL_COLUMN = {
    "TESS": "emotion",
    "CREMA-D": "major_emotion",
    "IEMOCAP": "emotion_4class",
    "RAVDESS": "labels",   # int, χρειάζεται μετατροπή μέσω label_names πριν το mapping
}

RAVDESS_LABEL_NAMES = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']


def filter_target_dataset(dataset_split, corpus, raw_label_column):
    label_map = LABEL_MAPS[corpus]

    def get_raw_label_str(example):
        val = example[raw_label_column]
        if corpus == "RAVDESS":
            return RAVDESS_LABEL_NAMES[val]   # int -> string
        return val   # ήδη string στα άλλα corpora

    def has_common_label(example):
        return label_map.get(get_raw_label_str(example)) is not None

    def add_common_label(example):
        raw_str = get_raw_label_str(example)
        example["common_label"] = label_map[raw_str]
        example["common_label_id"] = COMMON_LABEL2ID[example["common_label"]]
        return example

    filtered = dataset_split.filter(has_common_label)
    filtered = filtered.map(add_common_label)
    return filtered

In [7]:
# Cell 2 — επαναχρησιμοποίηση των loaders + label_mapping.py από πριν
# (LOADERS, RAW_LABEL_COLUMN, filter_target_dataset, κ.λπ. — ίδιος κώδικας με το inference_matrix.ipynb)

datasets_raw = {corpus: loader() for corpus, loader in LOADERS.items()}
filtered_targets = {
    corpus: filter_target_dataset(ds["test"], corpus, RAW_LABEL_COLUMN[corpus])
    for corpus, ds in datasets_raw.items()
}

Filter:   0%|          | 0/1400 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/540 [00:00<?, ? examples/s]

Map:   0%|          | 0/252 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1552 [00:00<?, ? examples/s]

Map:   0%|          | 0/1022 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1414 [00:00<?, ? examples/s]

Map:   0%|          | 0/1414 [00:00<?, ? examples/s]

In [8]:
# Cell 3 — εξαγωγή pooled embeddings ανά corpus
BATCH_SIZE = 16
MAX_PER_CORPUS = 500   # subsampling για λόγους υπολογιστικού κόστους στο MMD (βλ. σχόλιο παρακάτω)

def collate_fn(batch):
    audios = [ex["audio"]["array"] for ex in batch]
    inputs = base_feature_extractor(audios, sampling_rate=16000, padding=True, return_tensors="pt")
    return inputs

def extract_embeddings(dataset_split, model, feature_extractor, max_samples=None):
    if max_samples is not None and len(dataset_split) > max_samples:
        # τυχαίο subsampling με σταθερό seed για αναπαραγωγιμότητα
        rng = np.random.default_rng(42)
        idx = rng.choice(len(dataset_split), size=max_samples, replace=False)
        dataset_split = dataset_split.select(idx)

    loader = DataLoader(dataset_split, batch_size=BATCH_SIZE, collate_fn=collate_fn)
    device = next(model.parameters()).device

    all_embeddings = []
    with torch.no_grad():
        for inputs in loader:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            hidden = outputs.last_hidden_state          # (batch, time, 768)
            pooled = hidden.mean(dim=1)                  # mean-pooling πάνω στη χρονική διάσταση -> (batch, 768)
            all_embeddings.append(pooled.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)


embeddings = {}
for corpus, filtered_ds in filtered_targets.items():
    print(f"Εξαγωγή embeddings: {corpus} ({len(filtered_ds)} δείγματα, subsampled σε ≤{MAX_PER_CORPUS})")
    embeddings[corpus] = extract_embeddings(filtered_ds, base_model, base_feature_extractor, max_samples=MAX_PER_CORPUS)
    print(f"  -> shape: {embeddings[corpus].shape}")

# Αποθήκευση, ώστε να μη χρειαστεί ξανά GPU inference αν αλλάξεις κάτι στο MMD
EMB_DIR = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/30_cross_corpus/embeddings"
os.makedirs(EMB_DIR, exist_ok=True)
for corpus, emb in embeddings.items():
    np.save(os.path.join(EMB_DIR, f"{corpus}_embeddings.npy"), emb)

Εξαγωγή embeddings: TESS (800 δείγματα, subsampled σε ≤500)
  -> shape: (500, 768)
Εξαγωγή embeddings: RAVDESS (252 δείγματα, subsampled σε ≤500)
  -> shape: (252, 768)
Εξαγωγή embeddings: CREMA-D (1022 δείγματα, subsampled σε ≤500)
  -> shape: (500, 768)
Εξαγωγή embeddings: IEMOCAP (1414 δείγματα, subsampled σε ≤500)
  -> shape: (500, 768)


In [9]:
# Cell 4 — Maximum Mean Discrepancy (RBF kernel, unbiased estimator, median heuristic)
def rbf_mmd2(X, Y, gamma=None):
    """
    X: (m, d), Y: (n, d) — δύο σύνολα embeddings (m και n όχι απαραίτητα ίσα).
    Επιστρέφει MMD^2 (unbiased estimator) με RBF kernel.
    Το gamma (εύρος του πυρήνα) υπολογίζεται αυτόματα μέσω median heuristic
    αν δεν δοθεί ρητά: gamma = 1 / (2 * median(pairwise distances)^2)
    """
    XY = np.vstack([X, Y])
    if gamma is None:
        # median heuristic πάνω σε τυχαίο υποσύνολο ζευγών, για ταχύτητα
        n_sample = min(1000, len(XY))
        rng = np.random.default_rng(0)
        idx = rng.choice(len(XY), size=n_sample, replace=False)
        sq_dists = np.sum((XY[idx][:, None, :] - XY[idx][None, :, :]) ** 2, axis=-1)
        median_dist = np.median(sq_dists[sq_dists > 0])
        gamma = 1.0 / (2 * median_dist)

    def rbf_kernel_matrix(A, B):
        sq_dists = (np.sum(A**2, axis=1)[:, None] + np.sum(B**2, axis=1)[None, :] - 2 * A @ B.T)
        return np.exp(-gamma * sq_dists)

    Kxx = rbf_kernel_matrix(X, X)
    Kyy = rbf_kernel_matrix(Y, Y)
    Kxy = rbf_kernel_matrix(X, Y)

    m, n = len(X), len(Y)
    # unbiased estimator: αφαιρούμε τη διαγώνιο (i==j) από τα Kxx, Kyy
    sum_xx = (Kxx.sum() - np.trace(Kxx)) / (m * (m - 1))
    sum_yy = (Kyy.sum() - np.trace(Kyy)) / (n * (n - 1))
    sum_xy = Kxy.sum() / (m * n)

    return sum_xx + sum_yy - 2 * sum_xy

In [12]:
import pandas as pd

CORPORA = ["TESS", "RAVDESS", "CREMA-D", "IEMOCAP"]

# Cell 5 — Συμμετρικός πίνακας 4×4 MMD² μεταξύ όλων των ζευγών corpus
mmd_matrix = pd.DataFrame(index=CORPORA, columns=CORPORA, dtype=float)

for i, c1 in enumerate(CORPORA):
    mmd_matrix.loc[c1, c1] = 0.0   # απόσταση corpus από τον εαυτό του = 0
    for c2 in CORPORA[i+1:]:
        mmd2 = rbf_mmd2(embeddings[c1], embeddings[c2])
        mmd_matrix.loc[c1, c2] = mmd2
        mmd_matrix.loc[c2, c1] = mmd2   # συμμετρικό

print("=== MMD² μεταξύ corpora (encoder: μη fine-tuned data2vec) ===")
print(mmd_matrix.round(4))

=== MMD² μεταξύ corpora (encoder: μη fine-tuned data2vec) ===
           TESS  RAVDESS  CREMA-D  IEMOCAP
TESS     0.0000   0.5923   0.2585   0.2424
RAVDESS  0.5923   0.0000   0.2464   0.2706
CREMA-D  0.2585   0.2464   0.0000   0.0897
IEMOCAP  0.2424   0.2706   0.0897   0.0000
